# Mono-jet: inference with the released RACNN-ClassiFlow ensemble

This notebook loads the paper's RACNN-ClassiFlow-$\mathcal{L}$ ensemble (3 of 5 runs, `weights/ensembles.json`) and part of the mono-jet test set, and shows for two signal regions (one high-yield, one low-yield benchmark):

* the three input histograms ($p_T$, $E_T^\mathrm{miss}$, $\eta$) inside the SR cuts,
* the ensemble posterior $p(m_\mathrm{DM}, g_\chi \mid x)$ and its entropy $\hat H$,
* the classifier probability that signal is present.

It ends with a check against a stored paper number. The full test file has 2.9 million rows, so the notebook reads only the rows it needs.

**Before running:** download the weights and data with `python scripts/download.py` from the repository root.

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

REPO    = os.path.abspath("..")               # repository root (this notebook lives in monojet/)
DATA    = os.path.join(REPO, "data")          # Hugging Face dataset repo, see scripts/download.py
WEIGHTS = os.path.join(REPO, "weights")       # Hugging Face model repo
sys.path.insert(0, REPO)
import racnn_inference as ri

_, fd = ri.import_analysis("monojet")           # puts monojet/ on sys.path; fd = flexibledataset
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

import h5py

## Find the example rows without loading the whole test set

Each row is one SR: three 30-bin histograms inside the SR, 11 auxiliary inputs (event counts and cut/window edges), the true $\theta = (m_\mathrm{DM}, g_\chi)$ and the number of signal events in each histogram.

In [ ]:
PATH = f"{DATA}/monojet/test.h5"
scalers = fd.load_h5(PATH, index=[0]).output_scalers                  # scalers only
unnorm = lambda k, a: a * (scalers[k]["max"] - scalers[k]["min"] + 1e-8) + scalers[k]["min"]
with h5py.File(PATH) as f:
    mass = unnorm("MASS", f["flow_outputs/MASS"][()])
    coup = unnorm("COUP", f["flow_outputs/COUP"][()])
    n_sig = sum(f[f"mlp_outputs/{k}"][()] for k in ("PT_NEVENTS_SIG", "MET_NEVENTS_SIG", "ETA_NEVENTS_SIG"))
print(len(mass), "SR rows;", len(np.unique(np.round(mass) * 1e3 + np.round(coup, 2))), "distinct theta (incl. background-only)")

def best_row(m, g):
    """The SR with the most signal events for the theta point nearest to (m, g)."""
    d = np.hypot((mass - m) / 900, (coup - g) / 3); d[~np.isfinite(d)] = np.inf
    rows = np.where(np.isclose(d, d.min()))[0]
    return int(rows[np.argmax(n_sig[rows])])

examples = {"high yield": best_row(200, 2.85), "low yield": best_row(500, 1.1)}  # benchmarks of Figs 10 and 11
sub = fd.load_h5(PATH, index=sorted(examples.values()))
pos = {name: sorted(examples.values()).index(i) for name, i in examples.items()}
for name, i in examples.items():
    print(f"{name}: row {i}, m_DM = {mass[i]:.0f} GeV, g_chi = {coup[i]:.2f}, signal events = {n_sig[i]:.0f}")

## Posterior for the two examples

In [ ]:
ensemble = ri.load_ensemble("monojet", "racnnclassiflows", "best_model", WEIGHTS, device)
print("ensemble runs:", json.load(open(f"{WEIGHTS}/ensembles.json"))["monojet"]["racnnclassiflows"]["best_model"])
aux = {k: sub.unnormalize_input(k, sub.other_inputs[k]) for k in sub.other_inputs}

fig, ax = plt.subplots(2, 4, figsize=(18, 8), constrained_layout=True)
for r, (name, j) in enumerate(pos.items()):
    for c, (h, lab) in enumerate((("PT", r"$p_T$ [GeV]"), ("MET", r"$E_T^{miss}$ [GeV]"), ("ETA", r"$\eta$"))):
        counts = sub.unnormalize_input(h, sub.histograms[h][j])
        ax[r, c].stairs(counts, np.linspace(aux[f"{h}_LOW"][j], aux[f"{h}_UP"][j], len(counts) + 1), fill=True, alpha=0.6)
        ax[r, c].set(xlabel=lab, ylabel="events per bin", title=f"{name}: {h}", yscale="log" if h != "ETA" else "linear")
    x_cnn, x_aux, _, _ = (t.unsqueeze(0).double().to(device) for t in sub[j])
    g0, g1, dens, H = ri.posterior_grid(ensemble, sub, "monojet", x_cnn, x_aux,
                                        (scalers["MASS"]["min"], scalers["MASS"]["max"]), (scalers["COUP"]["min"], scalers["COUP"]["max"]), 80)
    _, p_sig = ri.classifier_probability(ensemble, x_cnn, x_aux)
    i = examples[name]
    ax[r, 3].pcolormesh(g0, g1, dens, shading="auto", cmap="viridis")
    ax[r, 3].plot(mass[i], coup[i], "r*", ms=14, label="true $\\theta$"); ax[r, 3].legend(loc="upper right")
    ax[r, 3].set(xlabel=r"$m_{DM}$ [GeV]", ylabel=r"$g_\chi$", title=f"$\\hat H$ = {H:.2f},  P(signal) = {float(p_sig):.3f}")
plt.show()

## Check against the paper

The fine-tuning test set (`finetune/test_pinmargin.h5`, Fig 15) is small, so it gives a quick check. The base model of Fig 15, RACNN-ClassiFlow-$\mathcal{L}$ run 5 before fine-tuning, has mean NLL $-0.567$ on it; the $\Delta$ checkpoint of run 4 has $-0.488$.

In [ ]:
ft_test = DataLoader(fd.load_h5(f"{DATA}/monojet/finetune/test_pinmargin.h5"), batch_size=1024)
for run, ckpt in ((5, "best_model"), (4, "best_model_ediff")):
    m = ri.load_model("monojet", "racnnclassiflows", run, ckpt, WEIGHTS, device)
    print(f"RACNN-ClassiFlow run {run} ({ckpt}): NLL {ri.mean_nll([m], ft_test, 'monojet', device):.3f}")